In [89]:
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

#### Get the flood dataset

In [90]:
# MERGED TWICE TO REMOVE DUPLICATES

flood_county = pd.read_csv(
    "../../02_processed_data/fema_flood_county_2000.csv"
).reset_index(drop=True)
flood_county.sort_values(by=["FIPS", "YEAR"]).dropna()
print(flood_county.info())
print(flood_county.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35450 entries, 0 to 35449
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   FIPS                   35450 non-null  int64  
 1   YEAR                   35450 non-null  int64  
 2   STATE                  35450 non-null  object 
 3   CZ_NAME                35450 non-null  object 
 4   COUNT                  35450 non-null  int64  
 5   TOTAL_DAMAGE_PROPERTY  35450 non-null  float64
 6   TOTAL_DURATION_HOURS   35450 non-null  float64
dtypes: float64(2), int64(3), object(2)
memory usage: 1.9+ MB
None
               FIPS          YEAR         COUNT  TOTAL_DAMAGE_PROPERTY  \
count  35450.000000  35450.000000  35450.000000           3.545000e+04   
mean   30434.366008   2010.019069     13.039126           3.367555e+06   
std    14880.622613      6.061159     18.316309           1.256051e+08   
min     1001.000000   2000.000000      1.000000           0.0

#### Get the HPI dataset

In [91]:
hpi_county = pd.read_csv("../../02_processed_data/hpi_county_2000.csv")
hpi_county.sort_values(by=["county_fips5", "yr"]).dropna()
print(hpi_county.info())
print(hpi_county.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21965 entries, 0 to 21964
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   cbsa_code                 21965 non-null  int64  
 1   cbsa_title                21965 non-null  object 
 2   county_fips5              21965 non-null  int64  
 3   county/county_equivalent  21965 non-null  object 
 4   state_name                21965 non-null  object 
 5   place_name                21965 non-null  object 
 6   place_id                  21965 non-null  int64  
 7   yr                        21965 non-null  int64  
 8   index_nsa                 21965 non-null  float64
 9   index_prev                21965 non-null  float64
 10  hpi_change                21965 non-null  float64
 11  hpi_yoy                   21965 non-null  float64
dtypes: float64(4), int64(4), object(4)
memory usage: 2.0+ MB
None
          cbsa_code  county_fips5      place_id       

#### Checking for missing counties

In [92]:
# hpi_county[hpi_county["cbsa_code"] == "10100"]
flood_county[flood_county["FIPS"] == "46045"]

,FIPS,YEAR,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS


In [93]:
min(hpi_county["county_fips5"]), max(hpi_county["county_fips5"])

(1001, 56025)

In [94]:
mask = ~hpi_county["county_fips5"].isin(flood_county["FIPS"])
hpi_no_flood = hpi_county.loc[mask, "county_fips5"].unique()
hpi_no_flood

array([ 2170, 41031,  9120,  9190,  2090,  9110,  9130, 15005,  9170,
        9180, 53059, 13173,  9140])

In [95]:
mask = ~flood_county["FIPS"].isin(hpi_county["county_fips5"])
flood_no_hpi = flood_county.loc[mask, "FIPS"].unique()
flood_no_hpi

array([ 1005,  1011,  1013, ..., 56041, 56043, 56045], shape=(2115,))

#### Merge flood and HPI datasets

In [96]:
hpi_flood = pd.merge(
    hpi_county[["county_fips5", "yr", "hpi_yoy", "index_nsa"]],
    flood_county,
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "YEAR"],
    how="left",
)

# Identify columns that came from flood_county (excluding join keys)
flood_cols = [c for c in flood_county.columns if c not in ["FIPS", "YEAR"]]

# Impute those flood variables as 0
hpi_flood[flood_cols] = hpi_flood[flood_cols].fillna(0)

# Drop join-key duplicates from the merged df
hpi_flood = hpi_flood.drop(columns=["FIPS", "YEAR"])

hpi_flood.head()

,county_fips5,yr,hpi_yoy,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS
0,48059,2000,1.226954,115.7100,0,0,0.0,0.0,0.00
1,48059,2001,4.070521,120.4200,0,0,0.0,0.0,0.00
2,48059,2002,2.850440,123.8525,TEXAS,CALLAHAN,2.0,807000.0,14.08
3,48059,2003,2.620052,127.0975,TEXAS,CALLAHAN,2.0,0.0,2.00
4,48059,2004,4.455241,132.7600,TEXAS,CALLAHAN,4.0,100000.0,4.18


In [97]:
# CHECKING FOR DUPLICATES
temp = (
    hpi_flood.value_counts(subset=["county_fips5", "yr"])
    .reset_index()
    .sort_values(by=["county_fips5", "yr"])
)
temp[temp["count"] > 1]

,county_fips5,yr,count


#### Get heat dataset

In [98]:
heat_county = pd.read_csv(
    "../../02_processed_data/county_heat_index_2000_2020_with_HSI.csv"
).reset_index(drop=True)
heat_county

,id,name,state,year,value,rank,mean_1901_2000,heat_index,rel_dev,abs_z_year,abs_pct_year,exceed_90,exceed_95,exceed_100,HSI
0,AL-001,Autauga County,Alabama,2000,93.8,125.0,90.9,0.400821,0.031903,1.316932,0.906059,3.8,0.0,0.0,0.546615
1,AL-001,Autauga County,Alabama,2001,88.9,18.0,90.9,0.041761,-0.022002,0.622524,0.720271,0.0,0.0,0.0,0.295779
2,AL-001,Autauga County,Alabama,2002,90.8,60.0,90.9,0.182546,-0.001100,0.838350,0.815985,0.8,0.0,0.0,0.378973
3,AL-001,Autauga County,Alabama,2003,88.0,3.0,90.9,-0.009982,-0.031903,0.611812,0.704963,0.0,0.0,0.0,0.277611
4,AL-001,Autauga County,Alabama,2004,88.3,11.0,90.9,0.016426,-0.028603,1.005642,0.811312,0.0,0.0,0.0,0.411246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65242,MD-037,St. Mary&,NaN,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65243,MD-037,St. Mary&,NaN,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65244,MD-037,St. Mary&,NaN,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65245,MD-037,St. Mary&,NaN,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
#### Get the fIPS codes for states

In [100]:
us_states_fips = pd.read_csv("../../01_original_data/us_states_fips.csv").reset_index(
    drop=True
)
us_states_fips.head()

,state,fips,longname
0,AL,1,Alabama
1,AK,2,Alaska
2,AZ,4,Arizona
3,AR,5,Arkansas
4,CA,6,California


#### Create FIPS codes in heat dataset

In [101]:
def process_fips(heat_county):
    county_split = str(heat_county).split("-")
    state = str(
        us_states_fips[us_states_fips["state"] == county_split[0]]["fips"].values[0]
    ).zfill(2)
    county = county_split[1]
    return state + county


heat_county["FIPS"] = heat_county["id"].apply(lambda x: process_fips(x))
heat_county["FIPS"] = heat_county["FIPS"].astype(int)
heat_county

,id,name,state,year,value,rank,mean_1901_2000,heat_index,rel_dev,abs_z_year,abs_pct_year,exceed_90,exceed_95,exceed_100,HSI,FIPS
0,AL-001,Autauga County,Alabama,2000,93.8,125.0,90.9,0.400821,0.031903,1.316932,0.906059,3.8,0.0,0.0,0.546615,1001
1,AL-001,Autauga County,Alabama,2001,88.9,18.0,90.9,0.041761,-0.022002,0.622524,0.720271,0.0,0.0,0.0,0.295779,1001
2,AL-001,Autauga County,Alabama,2002,90.8,60.0,90.9,0.182546,-0.001100,0.838350,0.815985,0.8,0.0,0.0,0.378973,1001
3,AL-001,Autauga County,Alabama,2003,88.0,3.0,90.9,-0.009982,-0.031903,0.611812,0.704963,0.0,0.0,0.0,0.277611,1001
4,AL-001,Autauga County,Alabama,2004,88.3,11.0,90.9,0.016426,-0.028603,1.005642,0.811312,0.0,0.0,0.0,0.411246,1001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65242,MD-037,St. Mary&,NaN,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24037
65243,MD-037,St. Mary&,NaN,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24037
65244,MD-037,St. Mary&,NaN,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24037
65245,MD-037,St. Mary&,NaN,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24037


In [102]:
# CHECKING FOR DUPLICATES
temp = (
    heat_county.value_counts(subset=["FIPS", "year"])
    .reset_index()
    .sort_values(by=["FIPS", "year"])
)
temp[temp["count"] > 1]

,FIPS,year,count


#### Merge heat and HPI/flood datasets

In [103]:
hpi_flood_heat = pd.merge(
    hpi_flood,
    heat_county[["FIPS", "year", "value", "heat_index", "HSI"]],
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "year"],
    how="left",
)

# Identify columns that came from heat_county, excluding join keys
heat_cols = ["FIPS", "year", "value", "heat_index", "HSI"]

# Impute only heat-related variables as 0
hpi_flood_heat[heat_cols] = hpi_flood_heat[heat_cols].fillna(0)

# Drop join-key duplicates
hpi_flood_heat = hpi_flood_heat.drop(columns=["FIPS", "year"])

hpi_flood_heat.head()
hpi_flood_heat

,county_fips5,yr,hpi_yoy,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS,value,heat_index,HSI
0,48059,2000,1.226954,115.7100,0,0,0.0,0.0,0.00,94.4,0.289948,0.509057
1,48059,2001,4.070521,120.4200,0,0,0.0,0.0,0.00,95.7,0.353297,0.587194
2,48059,2002,2.850440,123.8525,TEXAS,CALLAHAN,2.0,807000.0,14.08,90.8,0.059089,0.323792
3,48059,2003,2.620052,127.0975,TEXAS,CALLAHAN,2.0,0.0,2.00,92.8,0.178862,0.442478
4,48059,2004,4.455241,132.7600,TEXAS,CALLAHAN,4.0,100000.0,4.18,89.6,-0.000561,0.413233
...,...,...,...,...,...,...,...,...,...,...,...,...
21960,4027,2016,3.794342,171.2425,ARIZONA,YUMA,3.0,0.0,3.00,106.9,0.414770,0.872492
21961,4027,2017,3.854183,177.8425,0,0,0.0,0.0,0.00,107.2,0.419574,0.878639
21962,4027,2018,3.642268,184.3200,ARIZONA,YUMA,7.0,65000.0,17.83,106.4,0.402691,0.818690
21963,4027,2019,5.189345,193.8850,ARIZONA,YUMA,2.0,2000.0,7.87,106.7,0.410549,0.944888


In [104]:
temp = (
    hpi_flood_heat.value_counts(subset=["county_fips5", "yr"])
    .reset_index()
    .sort_values(by=["county_fips5", "yr"])
)
temp[temp["count"] > 1]

,county_fips5,yr,count


#### Merge the drought and wildfire datasets

In [105]:
drought_county = pd.read_csv(
    "../../02_processed_data/NCEI_annual_drought_county_FINAL.csv"
).reset_index(drop=True)
drought_county = drought_county[drought_county["FIPS"] != 0]
drought_county.sort_values(by=["FIPS", "Year"]).dropna()
drought_county = drought_county.drop(["County", "State", "State_abbr"], axis=1)

In [106]:
dupes = drought_county[drought_county.duplicated(subset=["FIPS", "Year"], keep=False)]
dupes = dupes.sort_values(by=["FIPS", "Year"])
drought_county = drought_county.drop_duplicates(subset=["FIPS", "Year"], keep="first")
drought_county.head()

,FIPS,Year,Annual_Mean_Index
0,1001,2000,-2.32
1,1001,2001,1.43
2,1001,2002,-0.76
3,1001,2003,2.39
4,1001,2004,0.00


In [107]:
drought_hpi_flood_heat = pd.merge(
    hpi_flood_heat,
    drought_county[["FIPS", "Year", "Annual_Mean_Index"]],
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "Year"],
    how="left",
)

drought_hpi_flood_heat = drought_hpi_flood_heat.drop(columns=["FIPS", "Year"])

drought_hpi_flood_heat["Annual_Mean_Index"] = drought_hpi_flood_heat[
    "Annual_Mean_Index"
].fillna(0)

drought_hpi_flood_heat.head()

,county_fips5,yr,hpi_yoy,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS,value,heat_index,HSI,Annual_Mean_Index
0,48059,2000,1.226954,115.7100,0,0,0.0,0.0,0.00,94.4,0.289948,0.509057,-3.36
1,48059,2001,4.070521,120.4200,0,0,0.0,0.0,0.00,95.7,0.353297,0.587194,0.39
2,48059,2002,2.850440,123.8525,TEXAS,CALLAHAN,2.0,807000.0,14.08,90.8,0.059089,0.323792,1.54
3,48059,2003,2.620052,127.0975,TEXAS,CALLAHAN,2.0,0.0,2.00,92.8,0.178862,0.442478,0.96
4,48059,2004,4.455241,132.7600,TEXAS,CALLAHAN,4.0,100000.0,4.18,89.6,-0.000561,0.413233,2.28


In [108]:
# CHECKING FOR DUPLICATES
temp = (
    drought_hpi_flood_heat.value_counts(subset=["county_fips5", "yr"])
    .reset_index()
    .sort_values(by=["county_fips5", "yr"])
)
temp[temp["count"] > 1]

,county_fips5,yr,count


In [109]:
wildfire_county = pd.read_csv(
    "../../02_processed_data/wildfire_data_preprocessed.csv"
).reset_index(drop=True)
wildfire_county.sort_values(by=["FIPS", "Year"]).dropna()
wildfire_county = wildfire_county.drop(["State"], axis=1)
wildfire_county.head()

,FIPS,Year,FIRE_FREQUENCY,TOTAL_FIRE_SIZE
0,1001,2003,43,274.8
1,1001,2004,86,744.5
2,1001,2005,63,234.2
3,1001,2006,62,534.7
4,1001,2007,93,477.3


In [110]:
wildfire_drought_hpi_flood_heat = pd.merge(
    drought_hpi_flood_heat,
    wildfire_county[["FIPS", "Year", "FIRE_FREQUENCY", "TOTAL_FIRE_SIZE"]],
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "Year"],
    how="left",
)

wildfire_drought_hpi_flood_heat = wildfire_drought_hpi_flood_heat.drop(
    columns=["FIPS", "Year"]
)

wildfire_drought_hpi_flood_heat["FIRE_FREQUENCY"] = wildfire_drought_hpi_flood_heat[
    "FIRE_FREQUENCY"
].fillna(0)
wildfire_drought_hpi_flood_heat["TOTAL_FIRE_SIZE"] = wildfire_drought_hpi_flood_heat[
    "TOTAL_FIRE_SIZE"
].fillna(0)

wildfire_drought_hpi_flood_heat.head()

,county_fips5,yr,hpi_yoy,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS,value,heat_index,HSI,Annual_Mean_Index,FIRE_FREQUENCY,TOTAL_FIRE_SIZE
0,48059,2000,1.226954,115.7100,0,0,0.0,0.0,0.00,94.4,0.289948,0.509057,-3.36,0.0,0.0
1,48059,2001,4.070521,120.4200,0,0,0.0,0.0,0.00,95.7,0.353297,0.587194,0.39,0.0,0.0
2,48059,2002,2.850440,123.8525,TEXAS,CALLAHAN,2.0,807000.0,14.08,90.8,0.059089,0.323792,1.54,0.0,0.0
3,48059,2003,2.620052,127.0975,TEXAS,CALLAHAN,2.0,0.0,2.00,92.8,0.178862,0.442478,0.96,1.0,100.0
4,48059,2004,4.455241,132.7600,TEXAS,CALLAHAN,4.0,100000.0,4.18,89.6,-0.000561,0.413233,2.28,19.0,61.0


In [111]:
# CHECKING FOR DUPLICATES
temp = (
    wildfire_drought_hpi_flood_heat.value_counts(subset=["county_fips5", "yr"])
    .reset_index()
    .sort_values(by=["county_fips5", "yr"])
)
temp[temp["count"] > 1]

,county_fips5,yr,count


#### Merge Drought/Wildfire/HPI/Flood/Heat/Hurricane datasets

In [112]:
storm_summary = pd.read_csv(
    "../../02_processed_data/storm_data_aggregated_county.csv", low_memory=False
)
storm_summary["year"] = storm_summary["year"].astype("int")
storm_summary["FIPS"] = storm_summary["FIPS"].astype("int")

huricane_grouped = storm_summary.groupby(["FIPS", "year"], as_index=False).agg(
    deaths_hurricane=("deaths", "sum"),
    injuries_hurricane=("injuries", "sum"),
    damage_hurricane=("damage", "sum"),
)

huricane_grouped.sort_values(by=["FIPS", "year"])

,FIPS,year,deaths_hurricane,injuries_hurricane,damage_hurricane
0,1003,2002,0.0,0.0,75000
1,1003,2004,0.0,0.0,0
2,1003,2005,0.0,0.0,0
3,1003,2020,2.0,0.0,236080000
4,1013,2004,0.0,0.0,0
...,...,...,...,...,...
947,51730,2003,1.0,0.0,9700000
948,51740,2003,0.0,0.0,10000000
949,51800,2003,0.0,0.0,33000000
950,51810,2003,0.0,0.0,39400000


In [113]:
wildfire_drought_hpi_flood_heat_huricane = pd.merge(
    wildfire_drought_hpi_flood_heat,
    huricane_grouped,
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "year"],
    how="left",
)

# Columns coming from huricane_grouped, excluding join keys
hurricane_cols = [c for c in huricane_grouped.columns if c not in ["FIPS", "year"]]

# Impute only hurricane variables as 0
wildfire_drought_hpi_flood_heat_huricane[hurricane_cols] = (
    wildfire_drought_hpi_flood_heat_huricane[hurricane_cols].fillna(0)
)


wildfire_drought_hpi_flood_heat_huricane

,county_fips5,yr,hpi_yoy,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS,value,heat_index,HSI,Annual_Mean_Index,FIRE_FREQUENCY,TOTAL_FIRE_SIZE,FIPS,year,deaths_hurricane,injuries_hurricane,damage_hurricane
0,48059,2000,1.226954,115.7100,0,0,0.0,0.0,0.00,94.4,0.289948,0.509057,-3.36,0.0,0.00,NaN,NaN,0.0,0.0,0.0
1,48059,2001,4.070521,120.4200,0,0,0.0,0.0,0.00,95.7,0.353297,0.587194,0.39,0.0,0.00,NaN,NaN,0.0,0.0,0.0
2,48059,2002,2.850440,123.8525,TEXAS,CALLAHAN,2.0,807000.0,14.08,90.8,0.059089,0.323792,1.54,0.0,0.00,NaN,NaN,0.0,0.0,0.0
3,48059,2003,2.620052,127.0975,TEXAS,CALLAHAN,2.0,0.0,2.00,92.8,0.178862,0.442478,0.96,1.0,100.00,NaN,NaN,0.0,0.0,0.0
4,48059,2004,4.455241,132.7600,TEXAS,CALLAHAN,4.0,100000.0,4.18,89.6,-0.000561,0.413233,2.28,19.0,61.00,NaN,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21960,4027,2016,3.794342,171.2425,ARIZONA,YUMA,3.0,0.0,3.00,106.9,0.414770,0.872492,-2.68,11.0,1326.60,NaN,NaN,0.0,0.0,0.0
21961,4027,2017,3.854183,177.8425,0,0,0.0,0.0,0.00,107.2,0.419574,0.878639,-2.05,17.0,103.00,NaN,NaN,0.0,0.0,0.0
21962,4027,2018,3.642268,184.3200,ARIZONA,YUMA,7.0,65000.0,17.83,106.4,0.402691,0.818690,-2.95,191.0,594.87,NaN,NaN,0.0,0.0,0.0
21963,4027,2019,5.189345,193.8850,ARIZONA,YUMA,2.0,2000.0,7.87,106.7,0.410549,0.944888,-0.23,64.0,1265.91,NaN,NaN,0.0,0.0,0.0


In [114]:
climate = wildfire_drought_hpi_flood_heat_huricane

# columns to drop
cols_to_drop = ["YEAR", "year_y", "State_fire", "State_ncei", "State_abbr", "Year", "year_x", "FIPS", "year", "value", "heat_index", "CZ_NAME"]

# drop them from df (ignore any that might be missing)
climate = climate.drop(columns=cols_to_drop, errors="ignore")

# quick check
print(climate.columns)
print(climate.info())
print(climate.describe())

Index(['county_fips5', 'yr', 'hpi_yoy', 'index_nsa', 'STATE', 'COUNT',
       'TOTAL_DAMAGE_PROPERTY', 'TOTAL_DURATION_HOURS', 'HSI',
       'Annual_Mean_Index', 'FIRE_FREQUENCY', 'TOTAL_FIRE_SIZE',
       'deaths_hurricane', 'injuries_hurricane', 'damage_hurricane'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21965 entries, 0 to 21964
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   county_fips5           21965 non-null  int64  
 1   yr                     21965 non-null  int64  
 2   hpi_yoy                21965 non-null  float64
 3   index_nsa              21965 non-null  float64
 4   STATE                  21965 non-null  object 
 5   COUNT                  21965 non-null  float64
 6   TOTAL_DAMAGE_PROPERTY  21965 non-null  float64
 7   TOTAL_DURATION_HOURS   21965 non-null  float64
 8   HSI                    21965 non-null  float64
 9   Annual_Mean_Index      

In [115]:
climate = climate.rename(
    columns={
        "COUNT": "FLOOD_FREQUENCY",
        "TOTAL_DAMAGE_PROPERTY": "FLOOD_PROPERTY_DAMAGE",
        "TOTAL_DURATION_HOURS": "FLOOD_DURATION_HOURS",
        "HSI": "HEAT_STRESS_INDEX",
        "Annual_Mean_Index": "DROUGHT_ANNUAL_MEAN_INDEX",
        "TOTAL_FIRE_SIZE": "FIRE_SIZE",
    }
)
climate.columns

Index(['county_fips5', 'yr', 'hpi_yoy', 'index_nsa', 'STATE',
       'FLOOD_FREQUENCY', 'FLOOD_PROPERTY_DAMAGE', 'FLOOD_DURATION_HOURS',
       'HEAT_STRESS_INDEX', 'DROUGHT_ANNUAL_MEAN_INDEX', 'FIRE_FREQUENCY',
       'FIRE_SIZE', 'deaths_hurricane', 'injuries_hurricane',
       'damage_hurricane'],
      dtype='object')

In [116]:
gdp = pd.read_csv("../../01_original_data/Financial Data/GDP_cleaned.csv")
gdp_small = gdp[["GeoFIPS", "Year", "Real_GDP"]]

In [117]:
climate_gdp = pd.merge(
    climate,
    gdp_small,
    left_on=["county_fips5", "yr"],
    right_on=["GeoFIPS", "Year"],
    how="left",
)
climate_gdp = climate_gdp.drop(columns=["GeoFIPS", "Year"])
climate_gdp.head()

,county_fips5,yr,hpi_yoy,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,FIRE_FREQUENCY,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP
0,48059,2000,1.226954,115.7100,0,0.0,0.0,0.00,0.509057,-3.36,0.0,0.0,0.0,0.0,0.0,NaN
1,48059,2001,4.070521,120.4200,0,0.0,0.0,0.00,0.587194,0.39,0.0,0.0,0.0,0.0,0.0,218398
2,48059,2002,2.850440,123.8525,TEXAS,2.0,807000.0,14.08,0.323792,1.54,0.0,0.0,0.0,0.0,0.0,232845
3,48059,2003,2.620052,127.0975,TEXAS,2.0,0.0,2.00,0.442478,0.96,1.0,100.0,0.0,0.0,0.0,224064
4,48059,2004,4.455241,132.7600,TEXAS,4.0,100000.0,4.18,0.413233,2.28,19.0,61.0,0.0,0.0,0.0,219285


In [118]:
unemployment = pd.read_csv("../../01_original_data/Financial Data/Unemployment2023_cleaned.csv")
unemployment_small = unemployment[["FIPS_Code", "Year", "Unemployment_Rate"]]

In [119]:
climate_gdp_unemployment = pd.merge(
    climate_gdp,
    unemployment_small,
    left_on=["county_fips5", "yr"],
    right_on=["FIPS_Code", "Year"],
    how="left",
)
climate_gdp_unemployment = climate_gdp_unemployment.drop(columns=["FIPS_Code", "Year"])
climate_gdp_unemployment.head()

,county_fips5,yr,hpi_yoy,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,FIRE_FREQUENCY,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP,Unemployment_Rate
0,48059,2000,1.226954,115.7100,0,0.0,0.0,0.00,0.509057,-3.36,0.0,0.0,0.0,0.0,0.0,NaN,3.9
1,48059,2001,4.070521,120.4200,0,0.0,0.0,0.00,0.587194,0.39,0.0,0.0,0.0,0.0,0.0,218398,4.3
2,48059,2002,2.850440,123.8525,TEXAS,2.0,807000.0,14.08,0.323792,1.54,0.0,0.0,0.0,0.0,0.0,232845,5.0
3,48059,2003,2.620052,127.0975,TEXAS,2.0,0.0,2.00,0.442478,0.96,1.0,100.0,0.0,0.0,0.0,224064,5.4
4,48059,2004,4.455241,132.7600,TEXAS,4.0,100000.0,4.18,0.413233,2.28,19.0,61.0,0.0,0.0,0.0,219285,4.7


In [120]:
property_tax = pd.read_csv("../../01_original_data/Financial Data/property_tax_cleaned.csv")


In [121]:
climate_gdp_unemployment_tax = pd.merge(
    climate_gdp_unemployment,
    property_tax,
    left_on=["county_fips5", "yr"],
    right_on=["fips", "year"],
    how="left",
)
climate_gdp_unemployment_tax = climate_gdp_unemployment_tax.drop(columns=["fips", "year"])
climate_gdp_unemployment_tax.head()

,county_fips5,yr,hpi_yoy,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,FIRE_FREQUENCY,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP,Unemployment_Rate,prop_rate
0,48059,2000,1.226954,115.7100,0,0.0,0.0,0.00,0.509057,-3.36,0.0,0.0,0.0,0.0,0.0,NaN,3.9,NaN
1,48059,2001,4.070521,120.4200,0,0.0,0.0,0.00,0.587194,0.39,0.0,0.0,0.0,0.0,0.0,218398,4.3,NaN
2,48059,2002,2.850440,123.8525,TEXAS,2.0,807000.0,14.08,0.323792,1.54,0.0,0.0,0.0,0.0,0.0,232845,5.0,NaN
3,48059,2003,2.620052,127.0975,TEXAS,2.0,0.0,2.00,0.442478,0.96,1.0,100.0,0.0,0.0,0.0,224064,5.4,2.43841
4,48059,2004,4.455241,132.7600,TEXAS,4.0,100000.0,4.18,0.413233,2.28,19.0,61.0,0.0,0.0,0.0,219285,4.7,2.41174


In [122]:
population = pd.read_parquet("../../01_original_data/population/Population_Data_2003_2019.parquet")
population_long = population.melt(
    id_vars=["FIPSCODE", "STNAME", "CTYNAME"], 
    value_vars=[col for col in population.columns if col.startswith("POPESTIMATE")],
    var_name="year", 
    value_name="population"
)
population_long["year"] = population_long["year"].str.extract("(\d+)").astype(int)
population_small = population_long[["FIPSCODE", "year", "population"]]
population_small.head()

<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
/var/folders/6q/jjdxqy5114dgzjyg60l84np80000gn/T/ipykernel_43120/4051454054.py:8: SyntaxWarning: invalid escape sequence '\d'
  population_long["year"] = population_long["year"].str.extract("(\d+)").astype(int)


,FIPSCODE,year,population
0,01000,2003,4503491.0
1,01001,2003,46800.0
2,01003,2003,151509.0
3,01005,2003,28594.0
4,01007,2003,21399.0


In [123]:
# Add zero to 4 digits FIPS Code
climate_gdp_unemployment_tax["county_fips5"] = climate_gdp_unemployment_tax["county_fips5"].astype(str).str.zfill(5)

climate_gdp_unemployment_tax_population = pd.merge(
    climate_gdp_unemployment_tax,
    population_small,
    left_on=["county_fips5", "yr"],
    right_on=["FIPSCODE", "year"],
    how="left",
)
climate_gdp_unemployment_tax_population = climate_gdp_unemployment_tax_population.drop(columns=["FIPSCODE", "year"])
climate_gdp_unemployment_tax_population.head()

,county_fips5,yr,hpi_yoy,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,FIRE_FREQUENCY,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP,Unemployment_Rate,prop_rate,population
0,48059,2000,1.226954,115.7100,0,0.0,0.0,0.00,0.509057,-3.36,0.0,0.0,0.0,0.0,0.0,NaN,3.9,NaN,NaN
1,48059,2001,4.070521,120.4200,0,0.0,0.0,0.00,0.587194,0.39,0.0,0.0,0.0,0.0,0.0,218398,4.3,NaN,NaN
2,48059,2002,2.850440,123.8525,TEXAS,2.0,807000.0,14.08,0.323792,1.54,0.0,0.0,0.0,0.0,0.0,232845,5.0,NaN,NaN
3,48059,2003,2.620052,127.0975,TEXAS,2.0,0.0,2.00,0.442478,0.96,1.0,100.0,0.0,0.0,0.0,224064,5.4,2.43841,12913.0
4,48059,2004,4.455241,132.7600,TEXAS,4.0,100000.0,4.18,0.413233,2.28,19.0,61.0,0.0,0.0,0.0,219285,4.7,2.41174,13156.0


In [124]:
climate_gdp_unemployment_tax_population

,county_fips5,yr,hpi_yoy,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,FIRE_FREQUENCY,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP,Unemployment_Rate,prop_rate,population
0,48059,2000,1.226954,115.7100,0,0.0,0.0,0.00,0.509057,-3.36,0.0,0.00,0.0,0.0,0.0,NaN,3.9,NaN,NaN
1,48059,2001,4.070521,120.4200,0,0.0,0.0,0.00,0.587194,0.39,0.0,0.00,0.0,0.0,0.0,218398,4.3,NaN,NaN
2,48059,2002,2.850440,123.8525,TEXAS,2.0,807000.0,14.08,0.323792,1.54,0.0,0.00,0.0,0.0,0.0,232845,5.0,NaN,NaN
3,48059,2003,2.620052,127.0975,TEXAS,2.0,0.0,2.00,0.442478,0.96,1.0,100.00,0.0,0.0,0.0,224064,5.4,2.43841,12913.0
4,48059,2004,4.455241,132.7600,TEXAS,4.0,100000.0,4.18,0.413233,2.28,19.0,61.00,0.0,0.0,0.0,219285,4.7,2.41174,13156.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21960,04027,2016,3.794342,171.2425,ARIZONA,3.0,0.0,3.00,0.872492,-2.68,11.0,1326.60,0.0,0.0,0.0,7284916,19.1,NaN,207247.0
21961,04027,2017,3.854183,177.8425,0,0.0,0.0,0.00,0.878639,-2.05,17.0,103.00,0.0,0.0,0.0,7543432,17.2,NaN,209507.0
21962,04027,2018,3.642268,184.3200,ARIZONA,7.0,65000.0,17.83,0.818690,-2.95,191.0,594.87,0.0,0.0,0.0,7522243,16.9,NaN,211612.0
21963,04027,2019,5.189345,193.8850,ARIZONA,2.0,2000.0,7.87,0.944888,-0.23,64.0,1265.91,0.0,0.0,0.0,7714421,16.6,NaN,213787.0
